### Exercise 1 - Blockchain Structure and Tamper Detection

In [ ]:
# Import libraries for hashing, JSON output, and timestamps.
import hashlib
import json
import time

# Return the SHA-256 hash of text.
def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Calculate one Merkle Root for all transactions.
def calculate_merkle_root(transactions):
    if not transactions:
        return sha256_text("")

    # Hash each transaction to start the Merkle tree.
    level = [sha256_text(tx) for tx in transactions]

    while len(level) > 1:
        # Duplicate the final hash when the count is odd.
        if len(level) % 2 == 1:
            level.append(level[-1])

        # Hash neighbouring pairs until one root remains.
        level = [
            sha256_text(level[i] + level[i + 1])
            for i in range(0, len(level), 2)
        ]

    return level[0]

# Store one block and its integrity information.
class Block:
    def __init__(self, index, transactions, previous_hash, timestamp=None):
        self.index = index
        self.timestamp = int(time.time()) if timestamp is None else timestamp
        self.transactions = list(transactions)
        self.previous_hash = previous_hash
        self.merkle_root = calculate_merkle_root(self.transactions)
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        # Hash the block header with SHA-256.
        header = {
            "index": self.index,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "merkle_root": self.merkle_root,
        }
        return hashlib.sha256(
            json.dumps(header, sort_keys=True).encode("utf-8")
        ).hexdigest()

    def as_dict(self):
        # Return readable block data for the output screenshots.
        return {
            "block": self.index,
            "transactions": self.transactions,
            "merkle_root": self.merkle_root,
            "previous_hash": self.previous_hash,
            "hash": self.hash,
        }


# Manage the blockchain and validate its integrity.
class Blockchain:
    def __init__(self):
        # Create the genesis block automatically.
        self.chain = [
            Block(
                index=1,
                transactions=["Genesis Block"],
                previous_hash="0" * 64,
            )
        ]

    def add_block(self, transactions):
        # Append a block linked to the current last block.
        previous_block = self.chain[-1]
        new_block = Block(
            index=len(self.chain) + 1,
            transactions=transactions,
            previous_hash=previous_block.hash,
        )
        self.chain.append(new_block)

    def validate(self):
        # Check every block and return the first failure.
        for position, block in enumerate(self.chain):
            # Check that transaction data matches the stored Merkle Root.
            recomputed_merkle = calculate_merkle_root(block.transactions)

            if block.merkle_root != recomputed_merkle:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Merkle Root mismatch",
                    "stored": block.merkle_root,
                    "recomputed": recomputed_merkle,
                    "blocks_checked": position + 1,
                }

            # Check that the stored block hash is still correct.
            recomputed_hash = block.calculate_hash()
            if block.hash != recomputed_hash:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Block hash mismatch",
                    "stored": block.hash,
                    "recomputed": recomputed_hash,
                    "blocks_checked": position + 1,
                }

            # Check the genesis rule or previous-block link.
            expected_previous = "0" * 64 if position == 0 else self.chain[position - 1].hash
            if block.previous_hash != expected_previous:
                return {
                    "valid": False,
                    "first_failing_block": block.index,
                    "reason": "Previous-hash link mismatch",
                    "stored": block.previous_hash,
                    "recomputed": expected_previous,
                    "blocks_checked": position + 1,
                }

        return {
            "valid": True,
            "first_failing_block": None,
            "reason": "No integrity errors detected",
            "stored": None,
            "recomputed": None,
            "blocks_checked": len(self.chain),
        }

    def print_chain(self):
        # Print each block for the required evidence.
        for block in self.chain:
            print(json.dumps(block.as_dict(), indent=2))
            print("-" * 88)


In [16]:
# Create 10 blocks: 1 genesis block plus 9 appended blocks.
blockchain = Blockchain()

# Different transaction data for Blocks 2 to 10.
transaction_sets = [
    ["Alice -> Bob: 5 BTC", "Bob -> Charlie: 1 BTC"],
    ["Charlie -> David: 2 BTC", "Eve -> Alice: 3 BTC"],
    ["David -> Alice: 0.5 BTC", "Bob -> Eve: 0.25 BTC"],
    ["Alice -> Eve: 1.2 BTC", "Charlie -> Bob: 0.8 BTC"],
    ["Eve -> David: 0.7 BTC", "David -> Bob: 0.4 BTC"],
    ["Bob -> Alice: 1.1 BTC", "Alice -> Charlie: 0.3 BTC"],
    ["Charlie -> Eve: 0.9 BTC", "Eve -> Bob: 0.2 BTC"],
    ["David -> Charlie: 1.5 BTC", "Bob -> David: 0.6 BTC"],
    ["Alice -> David: 0.75 BTC", "Eve -> Charlie: 0.45 BTC"],
]

# Add the nine non-genesis blocks.
for transactions in transaction_sets:
    blockchain.add_block(transactions)

assert len(blockchain.chain) == 10

# Print all 10 blocks in full JSON format.
print("=== BLOCKCHAIN BEFORE MODIFICATION ===")
blockchain.print_chain()

print("=== VALIDATION BEFORE MODIFICATION ===")
print(json.dumps(blockchain.validate(), indent=2))


=== BLOCKCHAIN BEFORE MODIFICATION ===
{
  "block": 1,
  "transactions": [
    "Genesis Block"
  ],
  "merkle_root": "89eb0ac031a63d2421cd05a2fbe41f3ea35f5c3712ca839cbf6b85c4ee07b7a3",
  "previous_hash": "0000000000000000000000000000000000000000000000000000000000000000",
  "hash": "0ea59b107dccb469eb4ee4feda09e51d741e5a589625204b22801f1f312df603"
}
----------------------------------------------------------------------------------------
{
  "block": 2,
  "transactions": [
    "Alice -> Bob: 5 BTC",
    "Bob -> Charlie: 1 BTC"
  ],
  "merkle_root": "b45848ab11dc09d63a0b2d1299aee371827e1edee3e4c847d0c7b519af4d8f3e",
  "previous_hash": "0ea59b107dccb469eb4ee4feda09e51d741e5a589625204b22801f1f312df603",
  "hash": "bc9b15c6bb82ac1c291a9529da0c0361d5201a229ee0693777b495ea8578c22b"
}
----------------------------------------------------------------------------------------
{
  "block": 3,
  "transactions": [
    "Charlie -> David: 2 BTC",
    "Eve -> Alice: 3 BTC"
  ],
  "merkle_root": "bb5704e0

In [17]:
# Modify Block 5 without updating its Merkle Root or hash.
fifth_block = blockchain.chain[4]
original_transaction = fifth_block.transactions[0]
fifth_block.transactions[0] = "ATTACKER -> ATTACKER: 999 BTC"

print("Original transaction:", original_transaction)
print("Modified transaction:", fifth_block.transactions[0])

# Show the modified block.
print("\n=== MODIFIED 5TH BLOCK ===")
print(json.dumps(fifth_block.as_dict(), indent=2))

# The validator should identify Block 5 as the first failure.
print("\n=== VALIDATION AFTER MODIFICATION ===")
tamper_result = blockchain.validate()
print(json.dumps(tamper_result, indent=2))


Original transaction: Alice -> Eve: 1.2 BTC
Modified transaction: ATTACKER -> ATTACKER: 999 BTC

=== MODIFIED 5TH BLOCK ===
{
  "block": 5,
  "transactions": [
    "ATTACKER -> ATTACKER: 999 BTC",
    "Charlie -> Bob: 0.8 BTC"
  ],
  "merkle_root": "ac54f23535b6c68e9d79e89e0fa9e20459cb859b059683d6a4b445e5a11c3d83",
  "previous_hash": "afac39c558a9ba4b831e4f02d83dbea97f54bba033b98a226083278ad9baac9f",
  "hash": "acf344da77f5c3933131cbe29766825c10d0c406f98cb767256b414633a5cc10"
}

=== VALIDATION AFTER MODIFICATION ===
{
  "valid": false,
  "first_failing_block": 5,
  "reason": "Merkle Root mismatch",
  "stored": "ac54f23535b6c68e9d79e89e0fa9e20459cb859b059683d6a4b445e5a11c3d83",
  "recomputed": "635e34751f436c0550d9bed99770f49f97b8ce091fa38e68626f9a714eabece4",
  "blocks_checked": 5
}
